In [1]:
print(" Installing dependencies...")
!pip install -q streamlit vllm qdrant-client FlagEmbedding sqlglot pydantic-settings python-dotenv pandas openpyxl requests pyngrok
print("Dependencies installed!")

 Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 

In [2]:
print(" Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

 Mounting Google Drive...
Mounted at /content/drive


In [3]:
MODEL_PATH = "/content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500"

# Verifying our model exists
import os
if os.path.exists(MODEL_PATH):
    print(f"Model found at: {MODEL_PATH}")
    # List model files
    files = os.listdir(MODEL_PATH)
    safetensors = [f for f in files if f.endswith('.safetensors')]
    print(f"   Found {len(safetensors)} safetensors files")
    if 'config.json' in files:
        print("  config.json found")
    if any('tokenizer' in f for f in files):
        print("  tokenizer files found")
else:
    print(f"ERROR: Model not found at {MODEL_PATH}")
    print(" Please check the path and update MODEL_PATH above")

Model found at: /content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500
   Found 4 safetensors files
  config.json found
  tokenizer files found


In [4]:
print("Uploading CleanSQL...")
from google.colab import files

# Uploading CleanSQL.zip
print("Please select CleanSQL.zip from your computer...")
uploaded = files.upload()

# Extract
!unzip -q CleanSQL.zip
%cd CleanSQL

print("CleanSQL ready!")

Uploading CleanSQL...
Please select CleanSQL.zip from your computer...


Saving CleanSQL.zip to CleanSQL.zip
/content/CleanSQL
CleanSQL ready!


In [5]:
%cd /content/CleanSQL

print(" Setting up ngrok...")
from pyngrok import ngrok

# REPLACE WITH YOUR NGROK TOKEN FROM https://dashboard.ngrok.com/
NGROK_TOKEN = "36K9aiLRt4fFFCQ9SnEb2paeTfd_67pwR3fWM61TJfmv13myR"

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()  # Kill any existing tunnels
print("Done, ngrok is configured!")


/content/CleanSQL
 Setting up ngrok...
Done, ngrok is configured!


In [6]:
# Check if our tokenizer is valid and proper

import json
import os

print(" Fixing tokenizer configuration...")

# Check current tokenizer_config.json
tokenizer_config_path = os.path.join(MODEL_PATH, "tokenizer_config.json")

if os.path.exists(tokenizer_config_path):
    print(f" Found tokenizer_config.json")

    # Read current config
    with open(tokenizer_config_path, 'r') as f:
        config = json.load(f)

    print(f"Current config keys: {list(config.keys())}")

    # Add missing model_type if not present
    if 'model_type' not in config:
        print(" 'model_type' missing - adding it")
        config['model_type'] = 'qwen2'

        # Write back
        with open(tokenizer_config_path, 'w') as f:
            json.dump(config, f, indent=2)

        print(" Fixed tokenizer_config.json")
    else:
        print(f" model_type already present: {config['model_type']}")
else:
    print(" tokenizer_config.json not found!")
    print("Creating a basic one...")

    # Create minimal tokenizer config
    basic_config = {
        "model_type": "qwen2",
        "tokenizer_class": "Qwen2Tokenizer"
    }

    with open(tokenizer_config_path, 'w') as f:
        json.dump(basic_config, f, indent=2)

    print(" Created tokenizer_config.json")

# Also check config.json
config_path = os.path.join(MODEL_PATH, "config.json")
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        model_config = json.load(f)

    if 'model_type' in model_config:
        print(f"config.json has model_type: {model_config['model_type']}")
    else:
        print(" config.json missing model_type - adding it")
        model_config['model_type'] = 'qwen2'
        with open(config_path, 'w') as f:
            json.dump(model_config, f, indent=2)
        print(" Fixed config.json")

print("\n Tokenizer configuration fixed! Try starting vLLM again.")



 Fixing tokenizer configuration...
 Found tokenizer_config.json
Current config keys: ['add_bos_token', 'add_prefix_space', 'added_tokens_decoder', 'additional_special_tokens', 'bos_token', 'clean_up_tokenization_spaces', 'eos_token', 'errors', 'extra_special_tokens', 'model_max_length', 'pad_token', 'split_special_tokens', 'tokenizer_class', 'unk_token', 'model_type']
 model_type already present: qwen2
config.json has model_type: qwen2

 Tokenizer configuration fixed! Try starting vLLM again.


In [7]:
print(" Starting vLLM server...")
print(" Model path:", MODEL_PATH)
print("="*60)

import subprocess
import sys
import time
import requests

# Kill any existing vLLM processes
!pkill -f vllm
time.sleep(2)

# Start vLLM
vllm_process = subprocess.Popen([
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_PATH,
    "--tokenizer", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--trust-remote-code",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--dtype", "auto",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.9",
    "--served-model-name", "qwen-coder"
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print(" Waiting for vLLM to start (checking every 15 seconds)...")
print(" Model loading from Google Drive takes ~5 minutes\n")

# Check every 15 seconds for up to 10 minutes
for i in range(40):
    time.sleep(15)
    elapsed = (i + 1) * 15

    try:
        response = requests.get("http://localhost:8000/health", timeout=2)
        if response.status_code == 200:
            print(f"\n vLLM is ready! (took {elapsed} seconds)")
            print(" Model loaded successfully!")
            break
    except:
        print(f"Time elapsed =>  {elapsed}s - Still loading...")
else:
    print("\n Timeout after 10 minutes")
    print("Check if process is still running:")
    !ps aux | grep vllm | grep -v grep


 Starting vLLM server...
 Model path: /content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500
 Waiting for vLLM to start (checking every 15 seconds)...
 Model loading from Google Drive takes ~5 minutes

Time elapsed =>  15s - Still loading...
Time elapsed =>  30s - Still loading...
Time elapsed =>  45s - Still loading...
Time elapsed =>  60s - Still loading...
Time elapsed =>  75s - Still loading...
Time elapsed =>  90s - Still loading...
Time elapsed =>  105s - Still loading...
Time elapsed =>  120s - Still loading...
Time elapsed =>  135s - Still loading...
Time elapsed =>  150s - Still loading...
Time elapsed =>  165s - Still loading...
Time elapsed =>  180s - Still loading...
Time elapsed =>  195s - Still loading...
Time elapsed =>  210s - Still loading...
Time elapsed =>  225s - Still loading...
Time elapsed =>  240s - Still loading...
Time elapsed =>  255s - Still loading...
Time elapsed =>  270s - Still loading...
Time elapsed =>  285s - Still loading...
Time elapsed =>  300

In [ ]:
# [OPTIONAL] Check if vLLM process crashed and get the error
print(" Checking vLLM process status...")
print("="*60)

if 'vllm_process' in globals():
    exit_code = vllm_process.poll()

    if exit_code is None:
        print(" vLLM process is still running")
    else:
        print(f" vLLM process exited with code: {exit_code}")

        # Read all error output
        print("\n FULL ERROR OUTPUT:")
        print("="*60)

        try:
            stderr = vllm_process.stderr.read().decode()
            stdout = vllm_process.stdout.read().decode()

            if stderr:
                print("STDERR:")
                print(stderr)

            if stdout:
                print("\nSTDOUT:")
                print(stdout)
        except:
            print("Could not read output")
else:
    print(" vllm_process not found")

# Also check system processes
print("\n System check:")
!ps aux | grep vllm | grep -v grep


In [ ]:
# Check if vllm is running, before trying to restart vLLM

# Check if vLLM is currently running
!ps aux | grep vllm | grep -v grep

# Try to connect
import requests
try:
    response = requests.get("http://localhost:8000/health", timeout=2)
    print(f" vLLM IS RUNNING! Status: {response.status_code}")

    # Test models endpoint
    models = requests.get("http://localhost:8000/v1/models", timeout=2)
    print(f" Models: {models.json()}")
except Exception as e:
    print(f" vLLM not running: {e}")



root        4997  9.2  1.9 13400372 1710664 ?    Sl   23:19   0:28 /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model /content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500 --tokenizer Qwen/Qwen2.5-Coder-7B-Instruct --trust-remote-code --host 0.0.0.0 --port 8000 --dtype auto --max-model-len 4096 --gpu-memory-utilization 0.9 --served-model-name qwen-coder
 vLLM IS RUNNING! Status: 200
 Models: {'object': 'list', 'data': [{'id': 'qwen-coder', 'object': 'model', 'created': 1764804283, 'owned_by': 'vllm', 'root': '/content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500', 'parent': None, 'max_model_len': 4096, 'permission': [{'id': 'modelperm-ad1c756d1f19d918', 'object': 'model_permission', 'created': 1764804283, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [8]:
# Check vLLM

print(" Verifying vLLM server health...")
import requests
import json

try:
    # Testing local health endpoint
    health_response = requests.get("http://localhost:8000/health", timeout=5)
    print(f" Health check: {health_response.status_code}")

    # Testing models endpoint
    models_response = requests.get("http://localhost:8000/v1/models", timeout=5)
    if models_response.status_code == 200:
        models_data = models_response.json()
        print(f" Models endpoint: {models_data}")

    # Testing a simple completion
    test_payload = {
        "model": "qwen-coder",
        "messages": [{"role": "user", "content": "Say 'Hello'"}],
        "max_tokens": 10
    }
    completion_response = requests.post(
        "http://localhost:8000/v1/chat/completions",
        json=test_payload,
        timeout=30
    )
    if completion_response.status_code == 200:
        result = completion_response.json()
        content = result['choices'][0]['message']['content']
        print(f" Test completion successful: '{content}'")
    else:
        print(f" Completion test failed: {completion_response.status_code}")
        print(f"   Response: {completion_response.text}")

    # Monitoring GPU usage
    print("\n GPU Status:")
    !nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

except Exception as e:
    print(f" vLLM verification failed: {e}")
    print(" vLLM may not be ready yet. Wait 30 more seconds and re-run this cell.")


 Verifying vLLM server health...
 Health check: 200
 Models endpoint: {'object': 'list', 'data': [{'id': 'qwen-coder', 'object': 'model', 'created': 1765616519, 'owned_by': 'vllm', 'root': '/content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500', 'parent': None, 'max_model_len': 4096, 'permission': [{'id': 'modelperm-bdd6cc6efa0a15ad', 'object': 'model_permission', 'created': 1765616519, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}
 Test completion successful: 'Hello! How can I assist you today?'

 GPU Status:
utilization.gpu [%], memory.used [MiB], memory.total [MiB]
60 %, 37651 MiB, 40960 MiB


In [ ]:
# [OPTIONAL] Check vLLM for errors

# Read the error output from the crashed process
print(" vLLM ERROR OUTPUT:")
print("="*60)

if 'vllm_process' in globals():
    # Try to read any remaining output
    try:
        stderr_output = vllm_process.stderr.read()
        stdout_output = vllm_process.stdout.read()

        if stderr_output:
            print("STDERR:")
            print(stderr_output.decode())
        else:
            print("No stderr captured")

        if stdout_output:
            print("\nSTDOUT:")
            print(stdout_output.decode())
        else:
            print("No stdout captured")
    except:
        print("Could not read process output (already consumed)")
else:
    print("vllm_process not found")



 vLLM ERROR OUTPUT:


Could not read process output (already consumed)


In [9]:
# Complete diagnostic
print("="*60)
print(" COMPLETE vLLM DIAGNOSTIC")
print("="*60)

# 1. Check process status
print("\n1️ Process Status:")
if 'vllm_process' in globals():
    if vllm_process.poll() is None:
        print("    vLLM process is running (PID exists)")
    else:
        print(f"    vLLM process exited with code: {vllm_process.poll()}")
else:
    print("    vllm_process variable not found")

# 2. Check system processes
print("\n2️ System Processes:")
!ps aux | grep vllm | grep -v grep

# 3. Check GPU
print("\n3️ GPU Status:")
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

# 4. Check port
print("\n4️ Port 8000:")
!netstat -tuln | grep 8000 || echo "Port 8000 not listening"

# 5. Try health check
print("\n5️ Health Check:")
import requests
try:
    response = requests.get("http://localhost:8000/health", timeout=2)
    print(f"    Health check passed: {response.status_code}")
except Exception as e:
    print(f"    Health check failed: {e}")

print("\n" + "="*60)





 COMPLETE vLLM DIAGNOSTIC

1️ Process Status:
    vLLM process is running (PID exists)

2️ System Processes:
root        6175  5.8  2.0 14334080 1759484 ?    Sl   08:54   0:27 /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model /content/drive/MyDrive/CleanSQL/Models/qwen2.5_cleansql_500 --tokenizer Qwen/Qwen2.5-Coder-7B-Instruct --trust-remote-code --host 0.0.0.0 --port 8000 --dtype auto --max-model-len 4096 --gpu-memory-utilization 0.9 --served-model-name qwen-coder

3️ GPU Status:
utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0 %, 37651 MiB, 40960 MiB

4️ Port 8000:
tcp        0      0 0.0.0.0:8000            0.0.0.0:*               LISTEN     

5️ Health Check:
    Health check passed: 200



In [10]:
# Upgrading qdrant-client to latest version
!pip install --upgrade qdrant-client

In [11]:
# Run Streamlit
import subprocess
import time
import requests
from pyngrok import ngrok

print("Cleaning up...")
ngrok.kill()
time.sleep(2)

# Verify vLLM is running locally
print("Checking vLLM on localhost:8000...")
try:
    response = requests.get("http://localhost:8000/health", timeout=2)
    print(f"vLLM is running: {response.status_code}")
except Exception as e:
    print(f"vLLM not accessible: {e}")
    raise Exception("vLLM must be running first")

# Configure .env to use LOCAL vLLM (not ngrok)
print("\nConfiguring .env for local vLLM...")
with open('.env', 'w') as f:
    f.write("""VLLM_HOST=127.0.0.1
VLLM_PORT=8000
VLLM_TIMEOUT=120.0
CLEANSQL_RAG_TOPK=3
CLEANSQL_ENABLE_RERANKER=false
CLEANSQL_EMBED_MODEL=BAAI/bge-m3
CLEANSQL_TEMP=0.2
CLEANSQL_SC_SAMPLES=3
""")
print(".env configured for localhost")

# Start Streamlit
print("\nStarting Streamlit...")
streamlit_process = subprocess.Popen([
    "streamlit", "run", "app_new.py",
    "--server.port", "8501",
    "--server.headless", "true"
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

time.sleep(5)

# Create only Streamlit tunnel
streamlit_tunnel = ngrok.connect(8501)
print(f"\nStreamlit tunnel: {streamlit_tunnel.public_url}")

print("\n" + "="*60)
print("CleanSQL is Running!")
print("="*60)
print(f"\nVISIT THIS URL: {streamlit_tunnel.public_url}")
print("\nHow it works:")
print("  - Streamlit runs on Colab (accessible via ngrok)")
print("  - vLLM runs on Colab (localhost only)")
print("  - Streamlit connects to vLLM locally on Colab")
print("  - You access Streamlit from your browser")
print("="*60)


Cleaning up...
Checking vLLM on localhost:8000...
vLLM is running: 200

Configuring .env for local vLLM...
.env configured for localhost

Starting Streamlit...

Streamlit tunnel: https://endocardial-filomena-omissively.ngrok-free.dev

CleanSQL is Running!

VISIT THIS URL: https://endocardial-filomena-omissively.ngrok-free.dev

How it works:
  - Streamlit runs on Colab (accessible via ngrok)
  - vLLM runs on Colab (localhost only)
  - Streamlit connects to vLLM locally on Colab
  - You access Streamlit from your browser


In [12]:
# Check tunnels

tunnels = ngrok.get_tunnels()
print(f"Total tunnels: {len(tunnels)}")
for tunnel in tunnels:
    print(f"  {tunnel.public_url} -> {tunnel.config['addr']}")



Total tunnels: 1
  https://endocardial-filomena-omissively.ngrok-free.dev -> http://localhost:8501
